In [1]:
import os
from dotenv import load_dotenv
from src.db import init_mongo

load_dotenv()
uri = os.getenv("MONGODB_URI")
mongo_client = init_mongo()
db = mongo_client["KB_PROPERTY_LAW"]

You successfully connected to MongoDB!


In [2]:
from src.triplet_extraction.llm import init_gpt

gpt_client = init_gpt()

## Generate Embeddings for Concepts Using OpenAI API

In [4]:
import json

concepts = db.concepts.find({})

with open("concept_embeddings.jsonl", "w", encoding="utf-8") as f:
    for concept in concepts:
        concept_id = str(concept["_id"])
        record = {
            "custom_id": f"{concept_id}_main",
            "method": "POST",
            "url": "/v1/embeddings",
            "body": {
                "model": "text-embedding-3-large",
                "input": concept["name"]
            }
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

        if concept["synonym"] is not None and len(concept["synonym"]) > 0:
            for idx, synonym in enumerate(concept["synonym"]):
                record = {
                    "custom_id": f"{concept['_id']}_syn_{idx}",
                    "method": "POST",
                    "url": "/v1/embeddings",
                    "body": {
                        "model": "text-embedding-3-large",
                        "input": synonym
                    }
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
file = gpt_client.files.create(
    file=open("embeddings.jsonl", "rb"),
    purpose="batch"
)

In [9]:
import json
import numpy as np
import faiss

JSONL_PATH = r"E:\Github\LawAssistant\notebook\test_new_retrival\batch_concept_embedding_output.jsonl"

vectors = []
vector_to_concept = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)

        if "response" not in r or r["response"]["status_code"] != 200:
            continue

        raw_id = r["custom_id"]
        concept_id = raw_id[:24]

        embedding = r["response"]["body"]["data"][0]["embedding"]

        vectors.append(embedding)
        vector_to_concept.append(concept_id)

X = np.array(vectors, dtype="float32")
# normalize for cosine similarity
faiss.normalize_L2(X)
dim = X.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(X)

print("FAISS index built")
print("Total vectors:", index.ntotal)

FAISS index built
Total vectors: 39280


In [10]:
import pickle

faiss.write_index(index, "concept.faiss")

with open("concept_id_map.pkl", "wb") as f:
    pickle.dump(vector_to_concept, f)

## Generate Embeddings for Sections Using OpenAI API

In [2]:
KEEP_SO_HIEU = [
    "31/2024/QH15",
    "101/2024/NĐ-CP",
    "102/2024/NĐ-CP",
    "103/2024/NĐ-CP",
    "226/2025/NĐ-CP",
    "27/2023/QH15",
    "95/2024/NĐ-CP",
    "29/2023/QH15",
    "96/2024/NĐ-CP",
    "91/2015/QH13",
    "52/2014/QH13"
]

In [3]:
# Load only what we need
docs = list(db.legal_sections.find({}))

# Collect all parent_ids that appear
parent_ids = {
    doc["parent_id"]
    for doc in docs
    if doc.get("parent_id") is not None
}

# Leaf nodes = ids that never appear as a parent_id
leaf_nodes = [
    doc for doc in docs
    if doc["_id"] not in parent_ids
]

print(f"Leaf nodes: {len(leaf_nodes)}")

Leaf nodes: 21894


In [ ]:
all_docs = list(
    db.legal_sections.find({}, {
        "_id": 1,
        "parent_id": 1,
        "title": 1,
        "type": 1,
        "content": 1,
        "so_hieu": 1,
        "full_path": 1
    })
)

id_map = {doc["_id"]: doc for doc in all_docs}

In [5]:
def collect_path(leaf):
    path = []
    current = leaf

    while current:
        path.append(current)
        pid = current.get("parent_id")
        current = id_map.get(pid)

    # reverse to get root → leaf
    path.reverse()
    return path

In [6]:
results = []

for leaf in leaf_nodes:
    path_nodes = collect_path(leaf)

    result = {
        "leaf_id": leaf["_id"],
        "so_hieu": leaf.get("so_hieu"),
        "full_path": leaf.get("full_path"),
        "combined_content": "\n".join(
            n["content"]
            for n in path_nodes
            if n.get("content")
        )
    }

    results.append(result)

In [7]:
for result in results[:10]:
    print("Section ID:", result["leaf_id"])
    print("So Hieu:", result["so_hieu"])
    print("Path:", result["full_path"])
    print("Combined Content:\n", result["combined_content"])
    print("-" * 40)

Section ID: b966689cda359c98430c4f6ce63db3bb5df63fb249c567c7e98bc23497bf1044
So Hieu: 31/2024/QH15
Path: 31/2024/QH15_chương i_điều 1
Combined Content:
 quy định chung
phạm vi điều chỉnh luật này quy định về chế độ sở hữu đất đai, quyền hạn và trách nhiệm của nhà nước đại diện chủ sở hữu toàn dân về đất đai và thống nhất quản lý về đất đai, chế độ quản lý và sử dụng đất đai, quyền và nghĩa vụ của công dân, người sử dụng đất đối với đất đai thuộc lãnh thổ của nước cộng hòa xã hội chủ nghĩa việt nam.
----------------------------------------
Section ID: e17097b833a43e5bc99e64c64eb8fce99e0ff56c328e24daaf2f420d412d07b1
So Hieu: 31/2024/QH15
Path: 31/2024/QH15_chương i_điều 2_khoản 1
Combined Content:
 quy định chung
đối tượng áp dụng
Cơ quan nhà nước thực hiện quyền hạn và trách nhiệm đại diện chủ sở hữu toàn dân về đất đai, thực hiện nhiệm vụ thống nhất quản lý nhà nước về đất đai.
----------------------------------------
Section ID: d4c6db29ccd0afcc94ac3c5e58a08169ff981cf435e9834e05f32f3b

In [8]:
import json
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

MAX_TOKENS = 3_000_000
current_tokens = 0
batch_index = 0

f = open(f"section_embeddings_{batch_index}.jsonl", "w", encoding="utf-8")

for r in results:
    text = r["combined_content"]
    tokens = len(enc.encode(text))

    if current_tokens + tokens > MAX_TOKENS:
        f.close()
        batch_index += 1
        current_tokens = 0
        f = open(f"sections_embedding/section_embeddings_{batch_index}.jsonl", "w", encoding="utf-8")

    record = {
        "custom_id": str(r["leaf_id"]),
        "method": "POST",
        "url": "/v1/embeddings",
        "body": {
            "model": "text-embedding-3-large",
            "input": text
        }
    }

    f.write(json.dumps(record, ensure_ascii=False) + "\n")
    current_tokens += tokens

f.close()

In [13]:
import os
import time
import json
from openai import OpenAI

gpt_client = OpenAI()

INPUT_SECTIONS_JSONL_FOLDER = r"E:\Github\LawAssistant\notebook\test_new_retrival\sections_embedding"
RESULT_FOLDER = r"E:\Github\LawAssistant\notebook\test_new_retrival\sections_embedding_result"

os.makedirs(RESULT_FOLDER, exist_ok=True)

def meta_path(filename):
    return os.path.join(RESULT_FOLDER, f"{filename}.meta.json")

def result_path(filename):
    return os.path.join(RESULT_FOLDER, f"{filename}.result.jsonl")

def process_file(filename):
    input_path = os.path.join(INPUT_SECTIONS_JSONL_FOLDER, filename)
    meta_file = meta_path(filename)
    result_file = result_path(filename)

    if os.path.exists(result_file):
        return

    if os.path.exists(meta_file):
        with open(meta_file, "r", encoding="utf-8") as f:
            meta = json.load(f)
        batch_id = meta["batch_id"]
        batch = gpt_client.batches.retrieve(batch_id)
    else:
        file_obj = gpt_client.files.create(
            file=open(input_path, "rb"),
            purpose="batch"
        )
        batch = gpt_client.batches.create(
            input_file_id=file_obj.id,
            endpoint="/v1/embeddings",
            completion_window="24h"
        )
        with open(meta_file, "w", encoding="utf-8") as f:
            json.dump(
                {"file_id": file_obj.id, "batch_id": batch.id},
                f,
                indent=2
            )

    while True:
        batch = gpt_client.batches.retrieve(batch.id)
        if batch.status in ("completed", "failed", "expired"):
            break
        print(f"Batch {batch.id} status: {batch.status}. Waiting 5 minutes...")
        time.sleep(300)

    if batch.status != "completed":
        return

    content = gpt_client.files.content(batch.output_file_id)
    with open(result_file, "wb") as f:
        f.write(content.read())

for filename in sorted(os.listdir(INPUT_SECTIONS_JSONL_FOLDER)):
    if not filename.endswith(".jsonl"):
        continue
    try:
        process_file(filename)
    except Exception:
        continue

Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47a status: in_progress. Waiting 5 minutes...
Batch batch_6973aded130081908460e0e0138bc47

In [15]:
import json
import numpy as np
import faiss
import pickle

JSONL_PATH = r"E:\Github\LawAssistant\notebook\test_new_retrival\sections_embedding_result"

vectors = []
vector_to_section = []

for filename in sorted(os.listdir(JSONL_PATH)):
    if not filename.endswith(".result.jsonl"):
        continue

    file_path = os.path.join(JSONL_PATH, filename)
    print(f"Processing {filename}")
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)

            if "response" not in r or r["response"]["status_code"] != 200:
                continue

            section_id = r["custom_id"]
            embedding = r["response"]["body"]["data"][0]["embedding"]

            vectors.append(embedding)
            vector_to_section.append(section_id)

X = np.array(vectors, dtype="float32")
# normalize for cosine similarity
faiss.normalize_L2(X)
dim = X.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(X)

print("FAISS index built")
print("Total vectors:", index.ntotal)

faiss.write_index(index, "sections.faiss")
with open("section_id_map.pkl", "wb") as f:
    pickle.dump(vector_to_section, f)

Processing section_embeddings_0.jsonl.result.jsonl
Processing section_embeddings_1.jsonl.result.jsonl
Processing section_embeddings_2.jsonl.result.jsonl
FAISS index built
Total vectors: 21883
